In [3]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Exchequer effect of the Budget 2026 ISA measures, £ million (negative = cost). Source: OBR EFO Oct 2026, policy tables 1.1.
xl = "../Data/efo-october-2026-detailed-forecast-tables-policy.xlsx"   # adjust path
raw = pd.read_excel(xl, sheet_name="1.1", header=None)

# year headers sit on the row containing "2026-27"; measures are found by name rather than row number
hdr = raw[raw.apply(lambda r: r.astype(str).str.contains("2026-27").any(), axis=1)].index[0]
cols = {c: str(v) for c, v in raw.loc[hdr].items() if str(v)[:2] == "20" and "-" in str(v)}
isa = raw[raw[3].astype(str).str.contains(r"\bISAs?\b", regex=True)]
assert len(isa) == 3, "expected cash ISA, First Time Buyer ISA and Lifetime ISA rows"

names = {"Cash ISAs": "Cash ISA limit kept at £20,000",
         "First Time Buyer ISA": "First Time Buyer ISA replaces LISA",
         "Lifetime ISA": "LISA withdrawal charge cut"}
rows = []
for _, r in isa.iterrows():
    label = next(v for k, v in names.items() if str(r[3]).startswith(k))
    for c, fy in cols.items():
        rows.append({"fy": fy, "measure": label, "value": float(r[c])})
d = pd.DataFrame(rows)

# saving first (teal), then the two costs (reds), so colours follow the story
order = ["First Time Buyer ISA replaces LISA", "Cash ISA limit kept at £20,000", "LISA withdrawal charge cut"]
net = d.groupby("fy", as_index=False)["value"].sum().assign(measure="Net effect")
wide = d.pivot(index="fy", columns="measure", values="value").reset_index()
wide["Net effect"] = net["value"].values

# sanity check against the Budget's Table 6.1 (£m, rounded): cash ISA row 11, FTB ISA row 38, LISA row 39
t61 = {"Cash ISA limit kept at £20,000": [0, -5, -20, -35, -50, -70],
       "First Time Buyer ISA replaces LISA": [0, 0, 0, 110, 110, 235],
       "LISA withdrawal charge cut": [-15, -20, -20, -25, -25, -25]}
for k, v in t61.items():
    got = wide[k].tolist()
    assert all(abs(a - b) < 6 for a, b in zip(got, v)), (k, got, v)

colours = ["#36B7B4", "#E6224B", "#F29BAE", "#676A86"]       # saving teal; costs red; net slate
scale = alt.Scale(domain=order + ["Net effect"], range=colours)
x = alt.X("fy:N", title=None, axis=alt.Axis(labelAngle=0))

bars = alt.Chart(d).mark_bar(size=46).encode(
    x=x,
    y=alt.Y("value:Q", title="£ million (negative = cost)", axis=alt.Axis(format=",.0f")),
    color=alt.Color("measure:N", sort=order, scale=scale,
        legend=alt.Legend(orient="bottom", title=None, labelLimit=400, symbolType="square",
                          columns=2, labelFontSize=10, symbolSize=60,
                          columnPadding=10, rowPadding=3)))

zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#c9c9c9").encode(y="y:Q")

netline = alt.Chart(net).encode(x=x, y="value:Q",
    color=alt.Color("measure:N", scale=scale, legend=None))
netmarks = netline.mark_line(strokeWidth=2.2) + netline.mark_point(filled=True, size=70, opacity=1)

# hover: shaded column, tooltip lists every series for that year
hover = alt.selection_point(fields=["fy"], on="mouseover", empty=False, clear="mouseout")
hit = alt.Chart(wide).mark_bar(size=110, color="#676A86").encode(
    x="fy:N",
    opacity=alt.condition(hover, alt.value(0.08), alt.value(0)),
    tooltip=[alt.Tooltip("fy:N", title="Fiscal year")] +
            [alt.Tooltip(f"{m}:Q", format="+,.0f", title=m + " (£m)") for m in order + ["Net effect"]]
).add_params(hover)

caption = alt.Title(
    text="Source: OBR, Economic and fiscal outlook – detailed forecast tables: policy (Table 1.1), October 2026",
    subtitle=["Budget 2026 ISA measures: effect on the Exchequer, £ million. Bars show each measure; the line is the net effect."],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (hit + zero + bars + netmarks).properties(
    width=400, height=300).configure(
    background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="isa_budget_2026", svg=True)
chart.save("isa_budget_2026.png", scale_factor=2.0)
chart

alt.LayerChart(...)